In [16]:
import argparse
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from dataloader.audio_data import ShipsEar, DeepShip, Whale
from models.ast_models import ASTModel
from models.pet_modules import ConvPass
from models.Adapter import AdapterBlock
from sklearn.metrics import roc_auc_score

In [17]:
"""
Arguments Parser
"""
def parse_options():
    parser = argparse.ArgumentParser(description="Adapter TICL")
    parser.add_argument('--gpu_id', type=str, default="cuda:0", help='the gpu id')
    opts = parser.parse_known_args()[0]
    torch.manual_seed(1234)
    opts.device = torch.device(opts.gpu_id)
    return opts


In [18]:
"""
Collate Function
"""
def af_pad_sequence(batch):
    # Make all tensor in a batch the same length by padding with zeros
    batch = [item.t() for item in batch]
    batch = torch.nn.utils.rnn.pad_sequence(batch, batch_first=True, padding_value=0.)
    return batch.permute(0, 2, 1)

In [19]:
def collate_fn(batch):
    spectrograms,labels = [], []
    # Gather in lists, and encode labels as indices
    for spec,label in batch:
        spectrograms += [spec]
        labels += [torch.tensor(label)]

    # Group the list of tensors into a batched tensor
    spectrograms = af_pad_sequence(spectrograms)
    labels = torch.stack(labels)
    return spectrograms,labels


In [20]:
def train_one_epoch(train_data_loader,model,optimizer,loss_fn,device):
    epoch_loss = []
    sum_correct_pred = 0
    total_samples = 0  
    model.train()
    ###Iterating over data loader
    for data, labels in train_data_loader:       
        #Loading data and labels to device
        data = data.to(device)
        labels = labels.to(device)       
        #Reseting Gradients
        optimizer.zero_grad()
        #Forward
        preds = model(data)
        #Calculating Loss
        _loss = loss_fn(preds, labels)
        epoch_loss.append(_loss.item())       
        #Backward
        _loss.backward()
        optimizer.step()

        sum_correct_pred += (torch.argmax(preds,dim=1) == labels).sum().item()
        total_samples += len(labels)
    acc = round(sum_correct_pred/total_samples,4)*100   
    ###Acc and Loss
    epoch_loss = np.mean(epoch_loss)   
    return epoch_loss, acc

In [21]:
def val_one_epoch(val_data_loader, model, loss_fn, device, num_class):
    epoch_loss = []
    sum_correct_pred = 0
    total_samples = 0
    model.eval()
    
    # 存储所有预测和标签以便计算AUC
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for data, labels in val_data_loader:
            data = data.to(device)
            labels = labels.to(device)
            preds = model(data)
            _loss = loss_fn(preds, labels)
            epoch_loss.append(_loss.item())
            
            # 计算正确预测的数量
            sum_correct_pred += (torch.argmax(preds, dim=1) == labels).sum().item()
            total_samples += len(labels)

            # 收集预测和标签
            all_preds.append(preds.cpu())  # 将预测移到CPU
            all_labels.append(labels.cpu())  # 将标签移到CPU

    # 计算总体损失和准确率
    epoch_loss = np.mean(epoch_loss)
    acc = round(sum_correct_pred / total_samples, 4) * 100

    # 将所有预测和标签合并为一个张量
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    # 计算AUC
    total_auc = 0
    for i in range(num_class):
        binary_labels = (all_labels == i).astype(int)  # 将标签转换为二进制形式
        if len(all_preds[all_labels == i]) > 0:  # 确保该类别有预测值
            auc = roc_auc_score(binary_labels, all_preds[:, i])
            total_auc += auc

    avg_auc = total_auc / num_class if num_class > 0 else 0

    return epoch_loss, acc, avg_auc

In [22]:
"""
Save weights - In Adapter Incremental setting - only pos embed, classifier weights and convolutional adapter weights need to be saved. 
"""
def save_model_weights(model, wt_name, save_mode):
    if save_mode == 'mlp':
        torch.save(model.mlp_head.state_dict(),wt_name)
    elif save_mode == 'pos':
        torch.save(model.v.pos_embed,wt_name)
    elif save_mode == 'adapter':
        weights = model.state_dict()
        # saving the adapter weights only!
        for name in weights:
            # isolating the convolutional adapter weights 
            if name.split('.')[1] == 'blocks':
                if name.split('.')[3] in ['conv1','conv2']:
                    # print(name)
                    continue
                else:
                    weights[name] = torch.zeros(1) # zero the non-adapter weights within encoder (reduce storage!)
            else:
                weights[name] = torch.zeros(1) # zero the non encoder layer weights (reduce storage!)
        torch.save(weights,wt_name) # 5.6 MB storage (full model, say ESC50, requires 336 MB storage)
    elif save_mode == 'full':
        torch.save(model.state_dict(),wt_name)

In [23]:
"""
load adapter weights only
"""
def load_adapter_weights(current_model, target_wt):

    current_wts = current_model.state_dict()

    for name in current_wts:
        # replacing convolutional adapter weights 
        if name.split('.')[1] == 'blocks':
            if name.split('.')[3] in ['conv1','conv2']:
                current_wts[name] = target_wt[name]
            else:
                continue # don't disturb other weights
        else:
            continue # don't disturb other weights
    current_model.load_state_dict(current_wts)
    return current_model


In [24]:
"""
Interpolate Pos Embed
"""
def interpolate_pos_embed(f_dim, t_dim, pos_embed):

  original_num_patches = 576 # 1 + 24*24 tokens
  original_embedding_dim =768
  oringal_hw = 24
  num_patches = f_dim * t_dim                     

  # get the positional embedding from vit model, skip the first cls tokens, reshape it to original 2D shape (24*24).                 original_pos_embed.shape:1,197,768
  new_pos_embed = pos_embed[:, 1:, :].detach().reshape(1, original_num_patches, original_embedding_dim).transpose(1, 2).reshape(1, original_embedding_dim, oringal_hw, oringal_hw)         #150528->1,768,196->1,196,768->1,768,14,1424

  # cut (from middle) or interpolate the second dimension of the positional embedding
  if t_dim <= oringal_hw:
      new_pos_embed = new_pos_embed[:, :, :, int(oringal_hw / 2) - int(t_dim / 2): int(oringal_hw / 2) - int(t_dim / 2) + t_dim]
  else:
      new_pos_embed = torch.nn.functional.interpolate(new_pos_embed, size=(oringal_hw, t_dim), mode='bilinear')

  # cut (from middle) or interpolate the first dimension of the positional embedding
  if f_dim <= oringal_hw:
      new_pos_embed = new_pos_embed[:, :, int(oringal_hw / 2) - int(f_dim / 2): int(oringal_hw / 2) - int(f_dim / 2) + f_dim, :]
  else:
      new_pos_embed = torch.nn.functional.interpolate(new_pos_embed, size=(f_dim, t_dim), mode='bilinear')

  # flatten the positional embedding
  new_pos_embed = new_pos_embed.reshape(1, original_embedding_dim, num_patches).transpose(1,2)
  # concatenate the above positional embedding with the cls token of the deit model.
  pos_embed = nn.Parameter(torch.cat([pos_embed[:, :1, :].detach(), new_pos_embed], dim=1))

  return pos_embed


In [25]:
"""
Main Loop
"""

opts = parse_options()
args=opts

In [27]:
model = ASTModel(label_dim=35, input_fdim=128, input_tdim=1001)
print("\t Model Loaded")

AssertionError: Please use timm == 0.4.5, the code might not be compatible with newer versions.

In [12]:
#####################################################################################################################################
# DeepShip dataset
#####################################################################################################################################
print("\n\nCurrent Dataset - DeepShip")
# DataLoader
train_anno = "./data/DeepShip_10s/protocols/train1.csv"
test_anno = "./data/DeepShip_10s/protocols/test1.csv"
train_dataset = DeepShip(train_anno, "./data/DeepShip_10s/audio/")
test_dataset = DeepShip(test_anno, "./data/DeepShip_10s/audio/")
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=16)
DeepShip_test_loader = test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=16)
print("\t Dataset Loaded")
print(train_dataset[0][0].shape)
train_dataset[0]



Current Dataset - DeepShip
	 Dataset Loaded
torch.Size([1, 128, 1001])


/home/zhuqian/anaconda3/envs/AICL/lib/python3.8/site-packages/torchaudio/functional/functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(
/home/zhuqian/anaconda3/envs/AICL/lib/python3.8/site-packages/torchaudio/functional/functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(


(tensor([[[-3.3144, -3.3144, -3.3144,  ..., -3.3144, -3.3144, -3.3144],
          [ 1.4957,  1.5420,  1.3857,  ...,  1.4051,  1.6050,  0.5216],
          [ 1.7639,  1.8102,  1.6539,  ...,  1.6733,  1.8732,  0.7898],
          ...,
          [ 0.9265,  0.3319,  0.2802,  ...,  0.3050,  0.1369,  0.4220],
          [ 0.9047,  0.2425,  0.2498,  ...,  0.1036,  0.2069,  0.5479],
          [ 0.9406,  0.3094,  0.2502,  ...,  0.1248,  0.0842,  0.5139]]]),
 4)

In [13]:
# first find the shape of patches f_dim and t_dim for DeepShip dataset. Utilize model.get_shape() for this
DeepShip_fdim, DeepShip_tdim = model.get_shape(fstride=10,tstride=10,input_fdim=128,input_tdim=1001)
#SSAST
#DeepShip_fdim, DeepShip_tdim = model.get_shape(fstride=10,tstride=10,input_fdim=128,input_tdim=1001,fshape=16,tshape=16)
print(DeepShip_fdim, DeepShip_tdim)
# replace adapters (the backbone still remains the same)
for i in range(12): model.v.blocks[i] = AdapterBlock(model.v.blocks[i], 32, DeepShip_fdim, DeepShip_tdim)
# replace classifier
model.mlp_head = nn.Linear(768,5)
# interpolate original timm pos-embed for Speech Commands
DeepShip_pos_embed = interpolate_pos_embed(f_dim=DeepShip_fdim, t_dim=DeepShip_tdim, pos_embed=torch.load(
    'vit_base_patch16_384_pos_embed.pth'))
DeepShip_pos_embed.requires_grad=False
model.v.pos_embed = DeepShip_pos_embed
# Since the model's backbone weights are frozen, they are unaffected, no matter whichever task you train!
model.unsqueeze=False
DeepShip_pos_embed.shape

12 99


torch.Size([1, 1189, 768])

In [14]:
model.to(args.device)
print("\t Model Loaded")
Adapter_params = sum(p.numel() for p in model.parameters() if p.requires_grad) - sum(p.numel() for p in model.mlp_head.parameters() if p.requires_grad)
Classifier_params = sum(p.numel() for p in model.mlp_head.parameters() if p.requires_grad)
print('\t DeepShip Adapter params = ',Adapter_params)
print('\t DeepShip Classifier params = ',Classifier_params)

	 Model Loaded
	 DeepShip Adapter params =  1198848
	 DeepShip Classifier params =  3845


In [15]:
CUDA_LAUNCH_BLOCKING=1
torch.cuda.empty_cache()

In [ ]:
# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr = 3e-4)
# Loss Function
loss_fn = nn.CrossEntropyLoss()

best_val_acc = []
print("\n\t Started Training")
for epoch in range(20):          
    ###Training
    loss, acc = train_one_epoch(train_loader,model,optimizer,loss_fn,args.device)
    ###Validation
    val_loss, val_acc,val_auc = val_one_epoch(test_loader,model,loss_fn,args.device, num_class=5)
    best_val_acc.append(val_acc)
    print(f"Epoch {epoch + 1}: Train Loss: {loss:.4f}, Train Acc: {acc:.2f}%, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%, Val AUC: {val_auc:.4f}")



	 Started Training
Epoch 1: Train Loss: 0.9986, Train Acc: 57.76%, Val Loss: 0.9383, Val Acc: 61.44%, Val AUC: 0.8856
Epoch 2: Train Loss: 0.8264, Train Acc: 65.94%, Val Loss: 1.0568, Val Acc: 58.69%, Val AUC: 0.8833
Epoch 3: Train Loss: 0.7568, Train Acc: 69.47%, Val Loss: 0.8911, Val Acc: 64.31%, Val AUC: 0.9167
Epoch 4: Train Loss: 0.6938, Train Acc: 72.54%, Val Loss: 0.6685, Val Acc: 73.07%, Val AUC: 0.9298
Epoch 5: Train Loss: 0.6431, Train Acc: 74.17%, Val Loss: 0.8140, Val Acc: 69.15%, Val AUC: 0.9262
Epoch 6: Train Loss: 0.5909, Train Acc: 76.76%, Val Loss: 0.6951, Val Acc: 73.20%, Val AUC: 0.9240
Epoch 7: Train Loss: 0.5435, Train Acc: 78.91%, Val Loss: 0.6676, Val Acc: 71.90%, Val AUC: 0.9329


+
.0

In [ ]:
"""
Eval 2
"""
#print("\n\t Auc of DeepShip dataset:",np.max(np.asarray(best_val_auc)))
print("\n\t Acc of DeepShip dataset:")
for acc in best_val_acc:
    print(f"\t\t {acc:.2f}")
# Save DeepShip task specific parameters
# save_model_weights(model,'WEIGHTS/MLP_adapter/DeepShip_classifier.pth',save_mode='mlp')
# save_model_weights(model,'WEIGHTS/MLP_adapter/DeepShip_pos_embed.pth',save_mode='pos')
# save_model_weights(model,'WEIGHTS/MLP_adapter/DeepShip_adapter.pth',save_mode='adapter') 
save_model_weights(model,'WEIGHTS/MLP_adapter/DeepShip_full.pth',save_mode='full')

In [ ]:
#####################################################################################################################################
# ShipsEar dataset
#####################################################################################################################################
print("\n\nCurrent Dataset - ShipsEar ")
# DataLoader
train_anno = "./data/ShipsEar/protocols/train.csv"
test_anno = "./data/ShipsEar/protocols/test.csv"
train_dataset = ShipsEar(train_anno, "./data/ShipsEar/audio/")
test_dataset = ShipsEar(test_anno, "./data/ShipsEar/audio/")
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=16)
ShipsEar_test_loader = test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=16)
print("\t Dataset Loaded")
print(train_dataset[0][0].shape)
print(test_dataset[0][1])

In [ ]:
# first find the shape of patches f_dim and t_dim for DeepShip dataset. Utilize model.get_shape() for this
ShipsEar_fdim, ShipsEar_tdim = model.get_shape(fstride=10,tstride=10,input_fdim=128,input_tdim=1001)
for p in model.v.parameters(): p.requires_grad=False
print(ShipsEar_fdim, ShipsEar_tdim)
# replace adapters (the backbone still remains the same)
for i in range(12): model.v.blocks[i] = AdapterBlock(model.v.blocks[i], 32,ShipsEar_fdim, ShipsEar_tdim)
# replace classifier
model.mlp_head = nn.Linear(768,5)
# interpolate original timm pos-embed for Speech Commands
ShipsEar_pos_embed = interpolate_pos_embed(f_dim=ShipsEar_fdim, t_dim=ShipsEar_tdim, pos_embed=torch.load(
    'vit_base_patch16_384_pos_embed.pth'))
ShipsEar_pos_embed.requires_grad=False
model.v.pos_embed = ShipsEar_pos_embed
# Since the model's backbone weights are frozen, they are unaffected, no matter whichever task you train!
model.unsqueeze=False
ShipsEar_pos_embed.shape

In [ ]:

model.to(args.device)
print("\t Model Loaded")
Adapter_params = sum(p.numel() for p in model.parameters() if p.requires_grad) - sum(p.numel() for p in model.mlp_head.parameters() if p.requires_grad)
Classifier_params = sum(p.numel() for p in model.mlp_head.parameters() if p.requires_grad)
print('\t ShipsEar Adapter params = ', Adapter_params)
print('\t ShipsEar Classifier params = ', Classifier_params)

In [ ]:
CUDA_LAUNCH_BLOCKING=1
# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr = 3e-4)
# Loss Function
loss_fn = nn.CrossEntropyLoss()
best_val_acc = []
best_val_auc = []

In [ ]:
print("\n\t Started Training")
optimizer.zero_grad()
for epoch in range(50):          
    ###Training
    loss, acc = train_one_epoch(train_loader,model,optimizer,loss_fn,args.device)
    ###Validation
    val_loss, val_acc, val_auc = val_one_epoch(test_loader,model,loss_fn,args.device,num_class=5)
    best_val_acc.append(val_acc)
    print(f"Epoch {epoch + 1}: Train Loss: {loss:.4f}, Train Acc: {acc:.2f}%, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%, Val AUC: {val_auc:.4f}")


In [ ]:


"""
Eval 2
"""
#print("\n\t Auc of ShipsEar dataset:",np.max(np.asarray(best_val_auc)))
print("\n\t Acc of ShipsEar dataset:")
for acc in best_val_acc:
    print(f"\t\t {acc:.2f}")
# Save DeepShip task specific parameters
save_model_weights(model,'WEIGHTS/adapter_incremental/ShipsEar_classifier.pth',save_mode='mlp')
save_model_weights(model,'WEIGHTS/adapter_incremental/ShipsEar_pos_embed.pth',save_mode='pos')
save_model_weights(model,'WEIGHTS/adapter_incremental/ShipsEar_adapter.pth',save_mode='adapter') 

In [ ]:
#  Inference on DeepShip
model.v.pos_embed = torch.load('WEIGHTS/adapter_incremental/ShipsEar_pos_embed.pth')
model.mlp_head = nn.Linear(768, 5)
model.mlp_head.load_state_dict(torch.load('WEIGHTS/adapter_incremental/ShipsEar_classifier.pth'))
# note that the attn weights and proj weights remain the same. 
for i in range(12): model.v.blocks[i] = AdapterBlock(model.v.blocks[i], 32,DeepShip_fdim, DeepShip_tdim)
model = load_adapter_weights(model, torch.load('WEIGHTS/adapter_incremental/ShipsEar_adapter.pth'))

In [ ]:

model.unsqueeze = False
model.f_dim, model.t_dim = DeepShip_fdim, DeepShip_tdim

model.to(args.device)
_, DeepShip_acc, Deepship_auc = val_one_epoch(DeepShip_test_loader, model, loss_fn, args.device, 5)
print("\n\t Acc on DeepShip after training on ShipsEar .....", DeepShip_acc)
print("\n\t Auc on DeepShip after training on ShipsEar .....", DeepShip_auc)

In [ ]:
#####################################################################################################################################
# Whale dataset
#####################################################################################################################################
print("\n\nCurrent Dataset - Whale Sound ")
# DataLoader
train_anno = "./data/whale_sound/protocols/new_train.csv"
test_anno = "./data/whale_sound/protocols/new_test.csv"
train_dataset = Whale(train_anno, "./data/whale_sound/pad_audio/")
test_dataset = Whale(test_anno, "./data/whale_sound/pad_audio/")
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=16)
Whale_test_loader = test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=16)
print("\t Dataset Loaded")


In [ ]:
# first find the shape of patches f_dim and t_dim for DeepShip dataset. Utilize model.get_shape() for this
whale_sound_fdim, whale_sound_tdim = model.get_shape(fstride=10,tstride=10,input_fdim=128,input_tdim=1001)

print(whale_sound_fdim, whale_sound_tdim)
# replace adapters (the backbone still remains the same)
for i in range(12): model.v.blocks[i] = AdapterBlock(model.v.blocks[i], 32,whale_sound_fdim, whale_sound_tdim )
# replace classifier
model.mlp_head = nn.Linear(768,16)
# interpolate original timm pos-embed for Speech Commands
whale_sound_pos_embed = interpolate_pos_embed(f_dim=whale_sound_fdim, t_dim=whale_sound_tdim, pos_embed=torch.load(
    'vit_base_patch16_384_pos_embed.pth'))
whale_sound_pos_embed.requires_grad=False
model.v.pos_embed = whale_sound_pos_embed
# Since the model's backbone weights are frozen, they are unaffected, no matter whichever task you train!
model.unsqueeze=False
whale_sound_pos_embed.shape

In [ ]:
model.to(args.device)
print("\t Model Loaded")
Adapter_params = sum(p.numel() for p in model.parameters() if p.requires_grad) - sum(p.numel() for p in model.mlp_head.parameters() if p.requires_grad)
Classifier_params = sum(p.numel() for p in model.mlp_head.parameters() if p.requires_grad)
print('\t whale_sound Adapter params = ', Adapter_params)
print('\t whale_sound Classifier params = ', Classifier_params)

In [ ]:
CUDA_LAUNCH_BLOCKING=1
# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr = 3e-4)
# Loss Function
loss_fn = nn.CrossEntropyLoss()
best_val_acc = []
best_val_auc = []
print("\n\t Started Training")
optimizer.zero_grad()
for epoch in range(50):          
    ###Training
    loss, acc = train_one_epoch(train_loader,model,optimizer,loss_fn,args.device)
    ###Validation
    val_loss, val_acc,val_auc= val_one_epoch(test_loader,model,loss_fn,args.device,num_class=16)
    best_val_acc.append(val_acc)
    print(f"Epoch {epoch + 1}: Train Loss: {loss:.4f}, Train Acc: {acc:.2f}%, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%, Val AUC: {val_auc:.4f}")

    

In [ ]:
"""
Eval 2
"""
#print("\n\t Auc of Whale dataset:",np.max(np.asarray(best_val_auc)))
print("\n\t Acc of Whale dataset:")
for acc in best_val_acc:
    print(f"\t\t {acc:.2f}")
# Save DeepShip task specific parameters
save_model_weights(model,'WEIGHTS/adapter_incremental/whale_sound_classifier.pth',save_mode='mlp')
save_model_weights(model,'WEIGHTS/adapter_incremental/whale_sound_pos_embed.pth',save_mode='pos')
save_model_weights(model,'WEIGHTS/adapter_incremental/whale_sound_adapter.pth',save_mode='adapter') 

In [ ]:
#  Inference on DeepShip
model.v.pos_embed = torch.load('WEIGHTS/MLP_adapter/DeepShip_pos_embed.pth')
model.mlp_head = nn.Linear(768, 5)
model.mlp_head.load_state_dict(torch.load('WEIGHTS/MLP_adapter/DeepShip_classifier.pth'))
# note that the attn weights and proj weights remain the same. 
for i in range(12): model.v.blocks[i] = AdapterBlock(model.v.blocks[i], 32,DeepShip_fdim,DeepShip_tdim)
model = load_adapter_weights(model, torch.load('WEIGHTS/MLP_adapter/DeepShip_adapter.pth'))

In [ ]:
model.unsqueeze = False
model.f_dim, model.t_dim = DeepShip_fdim, DeepShip_tdim

model.to(args.device)
_, DeepShip_acc, DeepShip_auc = val_one_epoch(DeepShip_test_loader, model, loss_fn, args.device,5)
print("\n\t Acc on DeepShip after training on Whale .....", DeepShip_acc)
print("\n\t Acc on DeepShip after training on Whale .....", DeepShip_auc)

In [ ]:
#  Inference on ShipsEar
model.v.pos_embed = torch.load('WEIGHTS/MLP_adapter/ShipsEar_pos_embed.pth')
model.mlp_head = nn.Linear(768, 5)
model.mlp_head.load_state_dict(torch.load('WEIGHTS/MLP_adapter/ShipsEar_classifier.pth'))
# note that the attn weights and proj weights remain the same. 
for i in range(12): model.v.blocks[i] = ConvPass(model.v.blocks[i], 32, ShipsEar_fdim, ShipsEar_tdim)
model = load_adapter_weights(model, torch.load('WEIGHTS/MLP_adapter/ShipsEar_adapter.pth'))

In [ ]:
model.unsqueeze = False
model.f_dim, model.t_dim = ShipsEar_fdim, ShipsEar_tdim

model.to(args.device)
_, ShipsEar_acc, ShipsEar_auc = val_one_epoch(ShipsEar_test_loader, model, loss_fn, args.device,5)
print("\n\t Acc on ShipsEar after training on Whale .....", ShipsEar_acc)
print("\n\t Acc on ShipsEar after training on Whale .....", ShipsEar_auc)